# Parameter and constraint investigation

This is a small, off-board calculation notebook. It does not configure hardware and it does not run an optimiser. Its purpose is to establish which quantities are fixed, which can be changed, and the first-order mathematical consequences of changing them.

In [ ]:
from pathlib import Path
import time
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'fpga' / 'fir_coeffs.txt').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'fpga' / 'fir_coeffs.txt').exists():
    raise RuntimeError('Run this notebook from inside the Test_Project checkout')

# Measured/fixed inputs. Replace F0 and Q0 with real-coil measurements.
FS = 125e6
F0 = 40e6
Q0 = 180.0
LOOP_DELAY = 119.87e-9
ADC_BITS = 14
SEO_SETPOINT = 1.0

def show(title, rows):
    print('\n' + title)
    for name, value, effect in rows:
        print('  %-25s %-18s %s' % (name, value, effect))

show('FIXED OR MEASURED INPUTS', [
    ('ADC sample rate', '125 MHz', 'sets every digital time/frequency scale'),
    ('coil f0', 'measured; ~40 MHz', 'sets the operating frequency; not adjustable here'),
    ('open-loop Q0', 'measured', 'sets B0=f0/Q0; not adjustable here'),
    ('input signal/noise', 'measured', 'sets attainable amplitude precision'),
    ('analogue gain/headroom', 'measured', 'sets SNR and clipping margin'),
    ('loop delay', '119.87 ns', 'rotates feedback phase versus frequency'),
    ('FPGA capacity/timing', 'XC7Z010 / 8 ns', 'limits CIC/FIR implementation'),
    ('SEO amplitude setpoint', 'experiment choice', 'target for software PI control'),
])
show('CHANGEABLE PARAMETERS', [
    ('CIC', 'R, stage count', 'rate, droop, alias rejection, word growth'),
    ('FIR', 'taps, bands, precision', 'droop correction, rejection, delay, cost'),
    ('FFT', 'length, window, averaging', 'amplitude bias/noise and record time'),
    ('resonator feedback', 'gain K, phase phi', 'Qeff or frequency shift'),
    ('SEO controller', 'Kp, Ki', 'settling, noise response, saturation'),
])
show('CALCULATED OUTCOMES', [
    ('amplitude error', 'bias and RMS', 'FFT measurement quality'),
    ('measurement time', 'Nfft R / fs', 'fastest possible control update'),
    ('Qeff', '(f0+df)/B', 'feedback outcome, never an input knob'),
    ('resonance shift', 'df', 'feedback outcome, never an input knob'),
    ('rejection at old f0', '20 log10(A/A0)', 'decoupling metric'),
    ('PI regulation error', 'RMS/overshoot', 'SEO amplitude stability'),
])
show('CONSTRAINTS', [
    ('aliasing', 'wanted band < fout/2', 'otherwise discarded signals fold in-band'),
    ('CIC overflow', 'sufficient Wacc', 'wrap cancels only with adequate width'),
    ('FIR throughput', '(L+1)/2+8 <= R', 'serial MAC must finish in one output period'),
    ('clipping', 'ADC/DAC/state headroom', 'distorts amplitude and resonance shape'),
    ('FPGA timing/resources', 'build must pass', 'hard implementation boundary'),
    ('feedback stability', 'B > 0', 'B<=0 is self-oscillation'),
    ('actuator saturation', 'gain rails', 'requires PI anti-windup'),
    ('control interval', 'record+compute+act', 'limits trackable amplitude changes'),
])
assert F0 == 40e6 and Q0 == 180.0  # later cells must not alter measured inputs

## CIC and FIR

For decimation by `R`, the output rate is `fs/R`. An `M`-stage CIC has response

$$|H(f)|=\left|\frac{\sin(\pi R f/f_s)}{R\sin(\pi f/f_s)}\right|^M,$$

and requires at least $W_{in}+M\log_2R$ accumulator bits. The existing symmetric FIR corrects the CIC droop. Its group delay is $(L-1)/(2f_{out})$, and its serial MAC must satisfy $(L+1)/2+8\le R$.

In [ ]:
FIR_Q = np.loadtxt(ROOT / 'fpga' / 'fir_coeffs.txt', comments='#', dtype=np.int64)
FIR_H = FIR_Q / float(1 << 17)
L = len(FIR_H)
WIN = 24

def cic_exact(f, R, stages):
    f = np.asarray(f, dtype=float)
    denominator = R * np.sin(np.pi * f / FS)
    answer = np.ones_like(f)
    nz = np.abs(denominator) > 1e-30
    answer[nz] = np.abs(np.sin(np.pi * R * f[nz] / FS) / denominator[nz]) ** stages
    return answer

def fir_response(u):
    n = np.arange(L) - (L - 1) / 2
    return np.abs(np.exp(-2j * np.pi * np.outer(u, n)) @ FIR_H)

cases = [(256, 3), (256, 4), (1024, 4), (4096, 4)]
u = np.linspace(0, 0.5, 401)
hf = fir_response(u)
print(' R     M    fout/kHz  Wacc  droop@0.2/dB  corrected/dB  FIR delay/ms  throughput')
for R, stages_count in cases:
    fout = FS / R
    hc = cic_exact(u * fout, R, stages_count)
    edge = np.argmin(np.abs(u - 0.2))
    droop = 20 * np.log10(max(hc[edge], 1e-15))
    corrected = 20 * np.log10(max(hc[edge] * hf[edge], 1e-15))
    width = WIN + stages_count * int(np.log2(R))
    delay_ms = 1e3 * (L - 1) / (2 * fout)
    valid = (L + 1) / 2 + 8 <= R
    print('%4d  %4d  %9.2f  %4d  %14.3f  %12.3f  %12.3f  %s' % (R, stages_count, fout/1e3, width, droop, corrected, delay_ms, valid))

assert abs(cic_exact(np.array([0.0]), 1024, 4)[0] - 1.0) < 1e-15
assert (L + 1) / 2 + 8 <= 256

fig, ax = plt.subplots(figsize=(7.5, 4))
hc = cic_exact(u * FS / 1024, 1024, 4)
ax.plot(u, 20*np.log10(np.maximum(hc, 1e-8)), label='CIC only')
ax.plot(u, 20*np.log10(np.maximum(hc*hf, 1e-8)), label='CIC + existing 309-tap FIR')
ax.axvline(0.2, color='0.5', ls=':', label='current passband edge')
ax.set(xlabel='frequency / output sample rate', ylabel='amplitude [dB]', ylim=(-80, 3), title='Existing correction filter; no coefficient redesign')
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout(); plt.show(); plt.close(fig)

## FFT amplitude: speed against accuracy

Herkenhoff used FPGA downmixing/decimation followed by a CPU FFT, maximum-bin search, and quadratic interpolation of that bin and its two neighbours. The software PID then adjusted feedback amplitude. The thesis states `R=4096`, 4096 samples, and a 12.5 Hz loop rate; the record-time arithmetic below checks whether those figures can coexist for this 125 MHz implementation.

$$\Delta f=f_{out}/N,\qquad T_{record}=N/f_{out},\qquad \hat A=2|X[k]|/\sum_nw[n].$$

In [ ]:
RNG = np.random.default_rng(2020)

def fft_amplitude(x, window_name):
    window = np.ones(len(x)) if window_name == 'rectangular' else np.hanning(len(x))
    spectrum = np.abs(np.fft.rfft(x * window))
    k = int(np.argmax(spectrum[1:])) + 1
    if not 1 <= k < len(spectrum) - 1:
        return 2 * spectrum[k] / window.sum()
    left, centre, right = spectrum[k-1:k+2]
    curvature = left - 2*centre + right
    delta = 0.0 if abs(curvature) < 1e-30 else 0.5 * (left - right) / curvature
    peak = centre - 0.25 * (left - right) * delta
    return 2 * peak / window.sum()

def tone_record(N, bin_offset, snr_db=None):
    n = np.arange(N)
    x = np.cos(2*np.pi*(19 + bin_offset)*n/N + 0.37)
    if snr_db is not None:
        noise_rms = (1/np.sqrt(2)) / (10**(snr_db/20))
        x = x + RNG.normal(0, noise_rms, N)
    return x

assert abs(fft_amplitude(tone_record(1024, 0), 'hann') - 1.0) < 2e-6
results = []
for R in (1024, 4096):
    for N in (1024, 4096):
        for window_name in ('rectangular', 'hann'):
            for offset in (0.0, 0.5):
                for snr_db in (10, 20, 40):
                    estimates = [fft_amplitude(tone_record(N, offset, snr_db), window_name) for _ in range(4)]
                    rms_error = float(np.sqrt(np.mean((np.asarray(estimates)-1.0)**2)))
                    results.append((N*R/FS, rms_error, R, N, window_name, offset, snr_db))

print('record ms   RMS error   R     N     window       offset  SNR/dB')
for row in results:
    if row[5] == 0.5 and row[6] == 20:
        print('%9.2f   %8.4f  %4d  %4d  %-11s %5.1f   %5d' % (1e3*row[0], row[1], row[2], row[3], row[4], row[5], row[6]))

thesis_record = 4096*4096/FS
print('\nThesis settings on this clock: record %.1f ms, record-only ceiling %.2f Hz; reported loop rate 12.5 Hz.' % (1e3*thesis_record, 1/thesis_record))
print('The discrepancy must be resolved from the original implementation before copying its numerical settings.')

fig, ax = plt.subplots(figsize=(7.5, 4))
for window_name, marker in [('rectangular', 'o'), ('hann', 's')]:
    selected_rows = [row for row in results if row[4] == window_name and row[6] == 20]
    ax.scatter([1e3*row[0] for row in selected_rows], [100*row[1] for row in selected_rows], marker=marker, label=window_name)
ax.set(xlabel='record time [ms]', ylabel='amplitude RMS error [%]', title='Small seeded FFT study; on-bin and half-bin tones at 20 dB SNR')
ax.grid(alpha=0.3); ax.legend(); fig.tight_layout(); plt.show(); plt.close(fig)

## Resonator feedback

The measured coil values remain fixed: $B_0=f_0/Q_0$. Feedback gain and phase produce new outcomes

$$B=B_0[1-K\cos(\phi-\phi_n)],\quad \Delta f=\frac{B_0K}{2}\sin(\phi-\phi_n),\quad Q_{eff}=\frac{f_0+\Delta f}{B}.$$

At the original resonance, $A/A_0=[1+(2\Delta f/B)^2]^{-1/2}$. Narrowing/high-Q and pure shifting are therefore separate phase settings, not one combined objective.

In [ ]:
B0 = F0 / Q0
phase_deg = np.linspace(-180, 180, 361)
phase_rad = np.deg2rad(phase_deg)
feedback_rows = []

fig, axes = plt.subplots(3, 1, figsize=(7.5, 8), sharex=True)
for K in (0.3, 0.7, 0.95, 1.05):
    bandwidth = B0 * (1 - K*np.cos(phase_rad))
    shift = 0.5 * B0 * K * np.sin(phase_rad)
    stable = bandwidth > 0
    qeff = np.where(stable, (F0 + shift)/bandwidth, np.nan)
    ratio = np.where(stable, 1/np.sqrt(1 + (2*shift/bandwidth)**2), np.nan)
    rejection_db = 20*np.log10(np.maximum(ratio, 1e-12))
    axes[0].plot(phase_deg, qeff/Q0, label='K=%.2f' % K)
    axes[1].plot(phase_deg, shift/B0)
    axes[2].plot(phase_deg, rejection_db)
    feedback_rows.append((K, float(np.nanmax(qeff/Q0)), float(np.nanmax(np.abs(shift/B0))), float(np.nanmin(rejection_db)), bool(np.any(~stable))))
axes[0].set(ylabel='Qeff / Q0', ylim=(0, 25), title='High-Q and pure-shift phases are distinct'); axes[0].legend()
axes[1].set(ylabel='shift / B0')
axes[2].set(xlabel='feedback phase [degree]', ylabel='amplitude change [dB]')
for ax in axes: ax.grid(alpha=0.3)
fig.tight_layout(); plt.show(); plt.close(fig)
print('K     max Q/Q0   max |shift|/B0   best rejection/dB   unstable')
for row in feedback_rows:
    print('%.2f   %8.2f   %14.3f   %17.2f   %s' % row)
assert F0 == 40e6 and Q0 == 180.0
assert feedback_rows[-1][-1]  # K>1 crosses B<=0 near the narrowing phase

## SEO amplitude PI

Following Herkenhoff, amplitude is estimated by the FFT and a software controller adjusts ion-feedback gain. This deliberately uses PI only: $e=A_{set}-\hat A$, $I_k=I_{k-1}+T_ue_k$, and $g=\mathrm{sat}(K_pe+K_iI)$. Integration pauses when saturation and error point in the same direction. The simple first-order amplitude plant below is for gain selection logic; measured plant dynamics must replace it before hardware tuning.

In [ ]:
TU = 0.08
GAIN_MAX = 1.2
NSTEP = 180

def run_pi(kp, ki, scenario, seed=7):
    rng = np.random.default_rng(seed)
    amplitude = np.zeros(NSTEP); command = np.zeros(NSTEP); target = np.ones(NSTEP)
    amplitude[0] = 0.55
    if scenario == 'step': target[:25] = 0.65
    integral = 0.0
    for k in range(1, NSTEP):
        plant_gain = 1.0 + (0.18*np.sin(2*np.pi*k/100) if scenario == 'drift' else 0.0)
        measured = amplitude[k-1] + (rng.normal(0, 0.035) if scenario == 'noise' else 0.0)
        error = target[k] - measured
        trial_integral = integral + TU*error
        raw = kp*error + ki*trial_integral
        clipped = float(np.clip(raw, 0, GAIN_MAX))
        if raw == clipped or (raw > GAIN_MAX and error < 0) or (raw < 0 and error > 0):
            integral = trial_integral
        command[k] = clipped
        equilibrium = plant_gain * command[k]
        amplitude[k] = amplitude[k-1] + TU*(equilibrium-amplitude[k-1])/0.7
    error_series = target - amplitude
    tail = error_series[60:]
    overshoot = max(0.0, float(np.max(amplitude-target)))
    rms = float(np.sqrt(np.mean(tail**2)))
    settled = np.nonzero(np.abs(error_series) < 0.05)[0]
    settling = float(settled[0]*TU) if len(settled) else np.inf
    saturation = float(np.mean((command <= 0) | (command >= GAIN_MAX)))
    return {'amplitude': amplitude, 'command': command, 'target': target, 'rms': rms, 'overshoot': overshoot, 'settling': settling, 'saturation': saturation}

print('Kp   Ki    RMS error  overshoot  settle/s  saturated')
pi_rows = []
for kp in (0.3, 0.7, 1.1):
    for ki in (0.2, 0.6, 1.0):
        trial = run_pi(kp, ki, 'noise')
        pi_rows.append((kp, ki, trial['rms'], trial['overshoot'], trial['settling'], trial['saturation']))
        print('%.1f  %.1f   %8.4f  %9.4f  %8.2f  %8.1f%%' % (kp, ki, trial['rms'], trial['overshoot'], trial['settling'], 100*trial['saturation']))

chosen = (0.7, 0.6)
fig, axes = plt.subplots(3, 1, figsize=(7.5, 7), sharex=True)
for ax, scenario in zip(axes, ('step', 'drift', 'noise')):
    trial = run_pi(chosen[0], chosen[1], scenario)
    t = TU*np.arange(NSTEP)
    ax.plot(t, trial['target'], 'k--', label='setpoint')
    ax.plot(t, trial['amplitude'], label='amplitude')
    ax.plot(t, trial['command'], alpha=0.7, label='feedback gain')
    ax.set_ylabel(scenario); ax.grid(alpha=0.3)
axes[0].legend(ncol=3); axes[-1].set_xlabel('time [s]')
fig.suptitle('PI examples, Kp=%.1f Ki=%.1f' % chosen); fig.tight_layout(); plt.show(); plt.close(fig)
assert np.max(run_pi(chosen[0], chosen[1], 'noise')['command']) <= GAIN_MAX
assert np.min(run_pi(chosen[0], chosen[1], 'noise')['command']) >= 0

## What must be measured next

1. Measure the real coil's $f_0$, $Q_0$, signal level, noise spectrum, and drift; do not optimise them.
2. Measure SEO amplitude error versus record time using the actual acquisition chain.
3. Measure resonator $Q_{eff}$ and shift versus feedback gain/phase to replace the normalized `K` model.
4. Measure SEO amplitude response to a small feedback-gain step to identify the PI plant time constant and gain.
5. Only then select CIC/FIR/FFT settings and tune PI gains.